# 📚 Knowledge Q&A Chatbot — GPT-2
### Assignment: Build a Chatbot using Hugging Face Transformers

**Model Used:** `gpt2-medium`  
**Approach:** Prompt-based text generation  
**Framework:** Hugging Face Transformers + PyTorch  

---

## 🎯 Objective
Build a console-based chatbot that answers user questions using a pre-trained GPT-2 transformer model via prompt engineering.

## 🧠 Learning Outcomes
- Understand how transformer-based language models work
- Use pre-trained NLP models from Hugging Face Model Hub
- Perform text generation using transformer models
- Build an interactive conversational system
- Understand prompt-based text generation

---
## 📦 Step 1 — Install Required Libraries

In [ ]:
# Install required packages
!pip install transformers torch

---
## 📥 Step 2 — Import Libraries

In [ ]:
# Core libraries
from transformers import pipeline
import textwrap
import re

print("✅ Libraries imported successfully!")

---
## 🤗 Step 3 — Load Pre-trained GPT-2 Model

We use Hugging Face's `pipeline` API to load `gpt2-medium` for text generation.

> **Note:** First run will download the model (~1.5 GB). Subsequent runs use the cached version.

In [ ]:
# ── Model Configuration ──────────────────────────────────────
MODEL_NAME     = "gpt2-medium"   # Pre-trained model from Hugging Face Hub
MAX_NEW_TOKENS = 130             # Max tokens to generate per response
WRAP_WIDTH     = 80              # Console output wrap width

print(f"🔄 Loading '{MODEL_NAME}' model... (first run may take a moment)\n")

# Load text-generation pipeline
# device=-1 means CPU; set device=0 to use GPU if available
generator = pipeline(
    "text-generation",
    model  = MODEL_NAME,
    device = -1,
)

print(f"✅ Model '{MODEL_NAME}' loaded successfully!")

---
## ✍️ Step 4 — Prompt Engineering

GPT-2 is a **completion model** — it continues whatever text you give it.  
We engineer a prompt that guides the model to behave like a Q&A assistant.

**Prompt structure:**
```
[System Context] + Question: <user question> + Answer:
```
The model then completes the `Answer:` portion.

In [ ]:
# ── System Context (Prompt Prefix) ───────────────────────────
# This tells GPT-2 to behave like a knowledgeable assistant
SYSTEM_CONTEXT = (
    "You are a knowledgeable and helpful AI assistant. "
    "Answer questions clearly, accurately and concisely.\n\n"
)

def build_prompt(question: str) -> str:
    """
    Constructs the full prompt from user question.
    GPT-2 will complete the 'Answer:' section.
    """
    return f"{SYSTEM_CONTEXT}Question: {question}\nAnswer:"

# ── Test the prompt builder ───────────────────────────────────
sample_prompt = build_prompt("What is Artificial Intelligence?")
print("📋 Sample Prompt:")
print("-" * 50)
print(sample_prompt)
print("-" * 50)

---
## 🔧 Step 5 — Response Generation & Post-Processing

In [ ]:
def extract_answer(generated: str, prompt: str) -> str:
    """
    Strips the prompt prefix from generated text
    and returns only the clean answer portion.
    """
    # Remove the prompt we passed in
    answer = generated[len(prompt):].strip()

    # Keep only the first 3 complete sentences
    sentences = re.split(r'(?<=[.!?])\s+', answer)
    clean = " ".join(sentences[:3]).strip()

    # Remove any incomplete trailing sentence
    if clean and clean[-1] not in ".!?":
        clean = clean.rsplit(".", 1)[0] + "." if "." in clean else clean

    return clean if clean else "I don't have enough information to answer that."


def get_response(user_question: str) -> str:
    """
    Full pipeline:
        1. Build prompt from user question
        2. Run GPT-2 text generation
        3. Extract and clean the answer
    """
    # Step 1 — Build prompt
    prompt = build_prompt(user_question)

    # Step 2 — Generate response
    result = generator(
        prompt,
        max_new_tokens       = MAX_NEW_TOKENS,
        num_return_sequences = 1,
        do_sample            = True,     # Enables sampling for diverse output
        top_k                = 40,       # Limits vocabulary to top 40 tokens
        top_p                = 0.90,     # Nucleus sampling threshold
        temperature          = 0.70,     # Controls randomness (lower = focused)
        repetition_penalty   = 1.4,      # Penalises repeated phrases
        pad_token_id         = 50256,    # GPT-2 EOS token ID
    )

    # Step 3 — Extract clean answer
    raw = result[0]["generated_text"]
    return extract_answer(raw, prompt)


print("✅ Response generation functions defined!")

---
## 🧪 Step 6 — Test the Model (Sample Queries)

Run a few sample queries to verify the model is working before starting the chat loop.

In [ ]:
# ── Quick Test ───────────────────────────────────────────────
test_questions = [
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What is machine learning?",
]

print("🧪 Running test queries...\n")
print("=" * 55)

for q in test_questions:
    print(f"You    : {q}")
    answer = get_response(q)
    wrapped = textwrap.fill(
        answer,
        width              = WRAP_WIDTH,
        initial_indent     = "Chatbot: ",
        subsequent_indent  = "         "
    )
    print(wrapped)
    print("-" * 55)

---
## 💬 Step 7 — Interactive Chatbot Loop

This is the main chatbot loop.

| Command | Action |
|---------|--------|
| Any text | Get an AI-generated answer |
| `exit` | End the conversation |
| `quit` | End the conversation |

> **Pipeline Flow:**  
> `User Input → Prompt Builder → GPT-2 Model → Response Extractor → Display → Loop`

In [ ]:
# ── Main Chatbot Loop ────────────────────────────────────────
def run_chatbot():
    print("=" * 55)
    print("   📚  Knowledge Q&A Bot  (GPT-2)")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI Knowledge Assistant.")
    print("         Ask me anything and I'll do my best to help!")
    print("         (type 'exit' or 'quit' to end)\n")

    while True:
        # ── Accept User Input ────────────────────────────────
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nChatbot: Goodbye! Keep exploring knowledge! 📖")
            break

        # ── Skip empty input ─────────────────────────────────
        if not user_input:
            continue

        # ── Exit Condition ───────────────────────────────────
        if user_input.lower() in {"exit", "quit"}:
            print("Chatbot: It was great helping you! Goodbye! 👋")
            break

        # ── Generate and Display Response ────────────────────
        response = get_response(user_input)

        wrapped = textwrap.fill(
            response,
            width             = WRAP_WIDTH,
            initial_indent    = "Chatbot: ",
            subsequent_indent = "         "
        )
        print(f"{wrapped}\n")


# ── Start the Chatbot ────────────────────────────────────────
run_chatbot()

---
## 📊 Step 8 — Pipeline Summary

```
┌─────────────┐     ┌──────────────────┐     ┌──────────────┐
│  User Input │ ──▶ │  Prompt Builder  │ ──▶ │  GPT-2 Model │
└─────────────┘     └──────────────────┘     └──────┬───────┘
                                                     │
                                                     ▼
┌──────────────┐     ┌──────────────────┐     ┌──────────────┐
│  Loop / Exit │ ◀── │  Display Output  │ ◀── │   Extractor  │
└──────────────┘     └──────────────────┘     └──────────────┘
```

## 🔑 Key Concepts Covered

| Concept | Implementation |
|---|---|
| Pre-trained model loading | `pipeline("text-generation", model="gpt2-medium")` |
| Prompt-based generation | `SYSTEM_CONTEXT + Question + Answer:` format |
| Text generation params | `top_k`, `top_p`, `temperature`, `repetition_penalty` |
| Post-processing | Regex sentence extraction, prompt stripping |
| Continuous conversation | `while True` loop with exit condition |
| Exit condition | Detects `exit` / `quit` keywords |

---
*Built with Hugging Face Transformers 🤗 | GPT-2 Medium Model*